# **DETECÇÃO DE FRAUDE EM CARTÃO DE CRÉDITO!**

**Problema:** identificar transações fraudulentas. Só 0,17% das transações são fraude, então acertar "quase tudo" não prova nada. A métrica principal é o **PR-AUC** (área sob a curva Precisão × Recall), com o ROC-AUC como referência.

**Roteiro do notebook**
1. Configuração e carregamento dos dados
2. Análise exploratória
3. Separação treino/teste e pré-processamento
4. Baseline: probabilidade de fraude só pelo valor (Bayes)
5. Ajuste do CatBoost (Optuna)
6. Comparação dos modelos por validação cruzada (dentro do treino)
7. Escolha do limite de alerta (sem usar o teste)
8. Avaliação final no teste
9. Importância das variáveis e conclusão

# **1. CONFIGURAÇÃO E CARREGAMENTO DOS DADOS**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

# Tema padrão dos gráficos (Seaborn)
# Default theme for all charts (Seaborn)
sns.set_theme(style="whitegrid")
PALETA = {"Normal": "tab:blue", "Fraude": "tab:red"}

# ----- Parâmetros do projeto (mude aqui, não no meio do código) -----
# ----- Project parameters (change them here, not in the middle of the code) -----
SEED = 42
N_FOLDS = 5                 # folds da validação cruzada / CV folds
PRECISAO_MINIMA = 0.80      # regra de negócio: >= 80% dos alertas devem ser fraude / business rule
REMOVER_DUPLICATAS = False  # True remove linhas idênticas antes do split / True drops identical rows before the split
RODAR_OPTUNA = False        # True refaz a busca de hiperparâmetros (demora) / True redoes the search (slow)
N_TRIALS = 20               # tentativas do Optuna, se RODAR_OPTUNA = True / Optuna trials if RODAR_OPTUNA = True

# Melhores parâmetros do CatBoost encontrados pelo Optuna na sua execução anterior
# Best CatBoost params found by Optuna in your previous run
PARAMS_CATBOOST = {
    "depth": 7,
    "learning_rate": 0.029617890239950373,
    "l2_leaf_reg": 1.2476394272569447,
    "random_strength": 4.7699849176399995,
    "bagging_temperature": 2.896896099223678,
    "scale_pos_weight": 1,
}

V_COLS = [f"V{i}" for i in range(1, 29)]   # variáveis anonimizadas (PCA) / anonymized variables (PCA)

#### Dataset Credit Card Fraud: 284.807 transações, das quais só 492 são fraude (baixe em https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud e coloque o `creditcard.csv` na mesma pasta do notebook). A proporção de fraude (o priori) é o ponto de partida do teorema de Bayes e o chute de referência das métricas.

In [ ]:
# Carrega os dados dentro da variável df
# Load the data into the df variable
df = pd.read_csv("creditcard.csv")
print("Formato:", df.shape)

# Linhas idênticas: o dataset não diz se são o mesmo registro duplicado ou coincidência
# Identical rows: the dataset does not say if they are the same record twice or coincidence
print("Linhas idênticas:", df.duplicated().sum(), "| fraudes entre elas:", df[df.duplicated()]["Class"].sum())
if REMOVER_DUPLICATAS:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicatas removidas. Novo formato:", df.shape)

# Contagem por classe e priori de fraude
# Count per class and fraud prior
print(df["Class"].value_counts())
print("Priori de fraude:", df["Class"].mean())

df.head()

# **2. ANÁLISE EXPLORATÓRIA: FRAUDE × NORMAL**

#### Quatro gráficos comparando transações normais e fraudes: quantidade por classe, distribuição dos valores (densidade), distribuição acumulada dos valores e transações ao longo do tempo.

In [ ]:
# Cópia só para gráficos, com rótulos legíveis e colunas transformadas
# Copy used only for plotting, with readable labels and transformed columns
df_plot = df.assign(
    Classe=df["Class"].map({0: "Normal", 1: "Fraude"}),
    log_valor=np.log1p(df["Amount"]),   # escala log: valores altos não dominam / log scale
    horas=df["Time"] / 3600,            # Time está em segundos / Time is in seconds
)

# Faixas definidas uma única vez, para normal e fraude usarem exatamente as mesmas
# Bins defined once, so normal and fraud use exactly the same ones
bins_valor = np.histogram_bin_edges(df_plot["log_valor"], bins=50)
bins_tempo = np.histogram_bin_edges(df_plot["horas"], bins=48)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1) Balanceamento das classes (escala log, senão a barra da fraude some)
# 1) Class balance (log scale, otherwise the fraud bar disappears)
ax = sns.countplot(data=df_plot, x="Classe", hue="Classe", order=["Normal", "Fraude"],
                   palette=PALETA, legend=False, ax=axes[0, 0])
ax.set_yscale("log")
for barras in ax.containers:
    ax.bar_label(barras)
ax.set_title("Quantidade por classe (escala log)")
ax.set_xlabel("")

# 2) Histograma dos valores, normalizado dentro de cada classe (densidade)
# 2) Amount histogram, normalized within each class (density)
sns.histplot(data=df_plot, x="log_valor", hue="Classe", bins=bins_valor, stat="density",
             common_norm=False, palette=PALETA, alpha=0.6, ax=axes[0, 1])
axes[0, 1].set_xlabel("log(1 + Amount)")
axes[0, 1].set_title("Distribuição dos valores")

# 3) Distribuição acumulada empírica F(x) = P(X <= x)
# 3) Empirical cumulative distribution F(x) = P(X <= x)
sns.ecdfplot(data=df_plot, x="log_valor", hue="Classe", palette=PALETA, ax=axes[1, 0])
axes[1, 0].set_xlabel("log(1 + Amount)")
axes[1, 0].set_ylabel("F(x) = P(X <= x)")
axes[1, 0].set_title("Distribuição acumulada dos valores")

# 4) Transações ao longo do tempo
# 4) Transactions over time
sns.histplot(data=df_plot, x="horas", hue="Classe", bins=bins_tempo, stat="density",
             common_norm=False, palette=PALETA, alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_xlabel("Horas desde a primeira transação")
axes[1, 1].set_title("Transações ao longo do tempo")

plt.tight_layout()
plt.show()

# **3. SEPARAÇÃO EM TREINO E TESTE E PRÉ-PROCESSAMENTO**

#### Treino (70%) e teste (30%), com a mesma proporção de fraude nos dois. **O teste fica guardado até a avaliação final**: toda escolha (modelo, hiperparâmetros, limite de alerta) usa só o treino. As entradas são V1 a V28 mais `Amount` e `Time` escalados com `RobustScaler`, que aprende a escala **só no treino** e depois é aplicado ao teste (assim o teste não influencia o pré-processamento).

In [ ]:
# Split primeiro, escala depois / Split first, scale afterwards
treino, teste = train_test_split(
    df, test_size=0.3, stratify=df["Class"], random_state=SEED
)
treino = treino.reset_index(drop=True)
teste = teste.reset_index(drop=True)

y_treino, y_teste = treino["Class"], teste["Class"]

# Entradas: V1..V28 + Amount e Time escalados (scaler aprende só com o treino)
# Inputs: V1..V28 + scaled Amount and Time (scaler learns from train only)
X_treino = treino[V_COLS].copy()
X_teste = teste[V_COLS].copy()
for col in ["Amount", "Time"]:
    scaler = RobustScaler()
    X_treino[f"scaled_{col.lower()}"] = scaler.fit_transform(treino[[col]])
    X_teste[f"scaled_{col.lower()}"] = scaler.transform(teste[[col]])

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)
print("Fraudes no treino:", y_treino.sum(), "| no teste:", y_teste.sum())
print("Fraude no treino:", y_treino.mean().round(5), "| no teste:", y_teste.mean().round(5))

# **4. BASELINE: PROBABILIDADE DE FRAUDE PELO VALOR (BAYES)**

#### Modelo probabilístico que usa só o valor da transação. As densidades e o priori são estimados só no treino. O score de cada transação do teste é a probabilidade de fraude (posteriori) da faixa de valor em que ela cai, calculada com probabilidade total e o teorema de Bayes. Serve de referência para os demais modelos.

In [ ]:
# Valor em escala log, para as faixas não ficarem dominadas por valores altos
# Amount on a log scale, so the bins are not dominated by large values
valor_treino = np.log1p(treino["Amount"])
valor_teste = np.log1p(teste["Amount"])

# Faixas e densidades estimadas só com o treino: P(valor | classe)
# Bins and densities estimated on train only: P(amount | class)
bins = np.histogram_bin_edges(valor_treino, bins=50)
f_fraude, _ = np.histogram(valor_treino[y_treino == 1], bins=bins, density=True)
f_normal, _ = np.histogram(valor_treino[y_treino == 0], bins=bins, density=True)
prior = y_treino.mean()

# Bayes: posteriori por faixa (faixas sem dados ficam com o priori)
# Bayes: posterior per bin (bins without data fall back to the prior)
num = f_fraude * prior                    # P(valor | fraude) * P(fraude)
den = num + f_normal * (1 - prior)        # probabilidade total P(valor) / total probability P(amount)
post = np.divide(num, den, out=np.full_like(num, prior), where=den > 0)

# Score de cada transação do teste = posteriori da faixa em que o valor dela cai
# Score of each test transaction = posterior of the bin its amount falls in
idx = np.clip(np.digitize(valor_teste, bins) - 1, 0, len(post) - 1)
score_valor = post[idx]

pr_valor = average_precision_score(y_teste, score_valor)
roc_valor = roc_auc_score(y_teste, score_valor)
print("PR-AUC:", pr_valor)
print("ROC-AUC:", roc_valor)
print("PR-AUC de um chute (priori):", y_teste.mean())

# **5. AJUSTE DO CATBOOST**

#### Os hiperparâmetros do CatBoost vêm de uma busca com Optuna que otimiza o PR-AUC em validação cruzada de 3 folds **dentro do treino**. A busca demora, então ela só roda se `RODAR_OPTUNA = True`; por padrão o notebook usa os parâmetros que você já encontrou. Depois, uma parada antecipada (com uma fatia do treino) define quantas iterações o modelo precisa.

In [ ]:
if RODAR_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    skf_busca = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    def objetivo(trial):
        # Espaço de busca / Search space
        p = {
            "depth": trial.suggest_int("depth", 4, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 30, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.5, 5),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 3),
            "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1, 5, 10, 20, 30]),
        }
        notas = []
        for idx_tr, idx_va in skf_busca.split(X_treino, y_treino):
            X_a, y_a = X_treino.iloc[idx_tr], y_treino.iloc[idx_tr]
            # Fatia só para a parada antecipada (não usa o fold de validação)
            # Slice only for early stopping (does not use the validation fold)
            X_f, X_es, y_f, y_es = train_test_split(X_a, y_a, test_size=0.15, stratify=y_a, random_state=SEED)
            m = CatBoostClassifier(iterations=2000, eval_metric="PRAUC", early_stopping_rounds=100,
                                   random_seed=SEED, verbose=0, **p)
            m.fit(X_f, y_f, eval_set=(X_es, y_es))
            s = m.predict_proba(X_treino.iloc[idx_va])[:, 1]
            notas.append(average_precision_score(y_treino.iloc[idx_va], s))
        return np.mean(notas)

    estudo = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    estudo.optimize(objetivo, n_trials=N_TRIALS)
    PARAMS_CATBOOST = estudo.best_params
    print("PR-AUC da melhor tentativa (CV 3 folds):", round(estudo.best_value, 4))

print("Parâmetros do CatBoost:", PARAMS_CATBOOST)

# Número de iterações por parada antecipada, usando só uma fatia do TREINO
# Number of iterations via early stopping, using only a slice of the TRAIN set
X_f, X_es, y_f, y_es = train_test_split(X_treino, y_treino, test_size=0.15, stratify=y_treino, random_state=SEED)
cb_es = CatBoostClassifier(iterations=2000, eval_metric="PRAUC", early_stopping_rounds=100,
                           random_seed=SEED, verbose=0, **PARAMS_CATBOOST)
cb_es.fit(X_f, y_f, eval_set=(X_es, y_es))
N_ITER = cb_es.get_best_iteration() + 1
print("Iterações escolhidas:", N_ITER)

# **6. COMPARAÇÃO DOS MODELOS POR VALIDAÇÃO CRUZADA**

#### Com só ~340 fraudes no treino, uma única divisão dá um resultado instável. Por isso comparo os modelos com validação cruzada estratificada **dentro do treino**: cada modelo é treinado 5 vezes e avaliado em dados que não viu. Reporto média e desvio padrão do PR-AUC. O Naive Bayes entra como referência simples, e Regressão Logística e Random Forest usam `class_weight="balanced"` para compensar o desbalanceamento.

In [ ]:
modelos = {
    "Naive Bayes": GaussianNB(),
    "Regressão Logística": make_pipeline(
        StandardScaler(),   # coloca as variáveis na mesma escala / puts variables on the same scale
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED),
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=SEED
    ),
    # CatBoost: iterações fixas (já definidas pela parada antecipada), sem eval_set
    # CatBoost: fixed iterations (already set by early stopping), no eval_set
    "CatBoost": CatBoostClassifier(iterations=N_ITER, random_seed=SEED, verbose=0, **PARAMS_CATBOOST),
}

# Validação cruzada estratificada: cada fold mantém a proporção de fraude
# Stratified CV: every fold keeps the fraud share
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def validar_cv(modelo, X, y):
    """Treina em N-1 folds e avalia no fold restante, N vezes.
    Retorna os scores "fora da amostra" (oof) e as métricas por fold.
    Trains on N-1 folds and evaluates on the remaining one, N times.
    Returns the out-of-fold scores (oof) and per-fold metrics."""
    oof = np.zeros(len(y))
    linhas = []
    for i, (idx_tr, idx_va) in enumerate(skf.split(X, y), start=1):
        m = clone(modelo)                       # cópia limpa do modelo / fresh copy of the model
        m.fit(X.iloc[idx_tr], y.iloc[idx_tr])
        s = m.predict_proba(X.iloc[idx_va])[:, 1]
        oof[idx_va] = s                         # guarda o score de quem ficou de fora / keep held-out scores
        linhas.append({"fold": i,
                       "PR-AUC": average_precision_score(y.iloc[idx_va], s),
                       "ROC-AUC": roc_auc_score(y.iloc[idx_va], s)})
    return oof, pd.DataFrame(linhas)


# Roda a validação cruzada de cada modelo (pode levar alguns minutos)
# Run CV for every model (may take a few minutes)
oof_scores, cv_folds = {}, {}
for nome, modelo in modelos.items():
    oof_scores[nome], cv_folds[nome] = validar_cv(modelo, X_treino, y_treino)
    print(f"{nome}: PR-AUC = {cv_folds[nome]['PR-AUC'].mean():.3f} "
          f"± {cv_folds[nome]['PR-AUC'].std():.3f}")

# Tabela resumo: média e desvio de cada métrica
# Summary table: mean and std of each metric
resumo_cv = pd.DataFrame({
    nome: {"PR-AUC (média)": f["PR-AUC"].mean(), "PR-AUC (desvio)": f["PR-AUC"].std(),
           "ROC-AUC (média)": f["ROC-AUC"].mean(), "ROC-AUC (desvio)": f["ROC-AUC"].std()}
    for nome, f in cv_folds.items()
}).T.round(4)
display(resumo_cv.sort_values("PR-AUC (média)", ascending=False))

# O melhor modelo é escolhido SÓ pela validação cruzada (nunca pelo teste)
# The best model is picked by CV ONLY (never by the test set)
melhor_modelo = resumo_cv["PR-AUC (média)"].idxmax()
print("Melhor modelo na validação cruzada:", melhor_modelo)

In [ ]:
# PR-AUC por fold: mostra a estabilidade de cada modelo
# PR-AUC per fold: shows how stable each model is
cv_long = pd.concat([f.assign(Modelo=nome) for nome, f in cv_folds.items()])

plt.figure(figsize=(9, 5))
sns.boxplot(data=cv_long, x="Modelo", y="PR-AUC", hue="Modelo", palette="Set2", legend=False)
sns.stripplot(data=cv_long, x="Modelo", y="PR-AUC", color="black", size=5, alpha=0.7)
plt.title(f"PR-AUC por fold ({N_FOLDS} folds, dentro do treino)")
plt.tight_layout()
plt.show()

#### **O `Time` ajuda?**

#### `Time` é o tempo desde a primeira transação, e os dados cobrem só dois dias. Uma variável assim pode ajudar de verdade ou só refletir particularidades dessa janela. Para saber, comparo o CatBoost com e sem `scaled_time` na mesma validação cruzada.

In [ ]:
# CatBoost sem a variável de tempo, na mesma validação cruzada
# CatBoost without the time variable, in the same cross-validation
_, cv_sem_time = validar_cv(modelos["CatBoost"], X_treino.drop(columns="scaled_time"), y_treino)

com, sem = cv_folds["CatBoost"]["PR-AUC"], cv_sem_time["PR-AUC"]
print(f"Com Time : PR-AUC = {com.mean():.3f} ± {com.std():.3f}")
print(f"Sem Time : PR-AUC = {sem.mean():.3f} ± {sem.std():.3f}")

dif, ruido = com.mean() - sem.mean(), max(com.std(), sem.std())
if abs(dif) <= ruido:
    print(f"Diferença de {dif:+.3f}, dentro do ruído entre folds ({ruido:.3f}): o Time não muda o resultado de forma clara.")
else:
    print(f"Diferença de {dif:+.3f}, acima do ruído entre folds ({ruido:.3f}).")

# **7. RECALL COM PRECISÃO MÍNIMA FIXA**

#### Fixei a precisão mínima dos alertas e medi quantas fraudes cada modelo consegue pegar (recall) em cada nível. Os scores usados são os da validação cruzada (dados que cada modelo não viu), então o teste continua intocado. Valor 0,00 significa que o modelo nunca chega àquela precisão.

In [ ]:
alvos = [0.1, 0.2, 0.3, 0.5, 0.8, 0.9]

linhas = {}
for nome, s in oof_scores.items():
    prec, rec, _ = precision_recall_curve(y_treino, s)
    # Maior recall possível mantendo a precisão acima de cada alvo
    # Highest possible recall while keeping precision above each target
    linhas[nome] = {f"precisão >= {a:.0%}": (rec[prec >= a].max() if (prec >= a).any() else 0.0)
                    for a in alvos}

recall_por_precisao = pd.DataFrame(linhas).T.round(2)
display(recall_por_precisao)

plt.figure(figsize=(9, 4))
sns.heatmap(recall_por_precisao, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Recall (fraudes pegas) para cada precisão mínima")
plt.tight_layout()
plt.show()

# **8. ESCOLHA DO LIMITE DE ALERTA (SEM USAR O TESTE)**

#### O limite de alerta é uma decisão de negócio. Aqui ele é escolhido com os scores da validação cruzada do melhor modelo: primeiro uma tabela com vários limites (para ver o custo de cada um), depois a regra "pegar o máximo de fraudes mantendo pelo menos `PRECISAO_MINIMA` dos alertas corretos".

In [ ]:
s_oof = oof_scores[melhor_modelo]


def tabela_limites(y, s, limites):
    """Para cada limite, conta alertas, acertos, falsos e fraudes perdidas.
    For each threshold, counts alerts, hits, false alarms and missed fraud."""
    linhas = []
    for limite in limites:
        # Score acima do limite gera alerta (1); abaixo, não (0)
        # Score above the threshold raises an alert (1); below, it does not (0)
        previsto = (s >= limite).astype(int)
        # vn = normais sem alerta, fp = alertas falsos, fn = fraudes que passaram, vp = fraudes pegas
        # vn = quiet normals, fp = false alerts, fn = missed fraud, vp = caught fraud
        vn, fp, fn, vp = confusion_matrix(y, previsto, labels=[0, 1]).ravel()
        linhas.append({
            "limite": limite,
            "alertas dados": vp + fp,
            "fraudes pegas (vp)": vp,
            "alertas falsos (fp)": fp,
            "fraudes que passaram (fn)": fn,
            "precisão": round(vp / (vp + fp), 2) if (vp + fp) else 0.0,
            "recall": round(vp / (vp + fn), 2),
        })
    return pd.DataFrame(linhas).set_index("limite")


print(f"Limites avaliados na validação cruzada ({melhor_modelo}):")
display(tabela_limites(y_treino, s_oof, [0.1, 0.2, 0.3, 0.5, 0.7]))

# Regra de negócio: entre os limites com precisão >= PRECISAO_MINIMA, o de maior recall
# Business rule: among thresholds with precision >= PRECISAO_MINIMA, the one with highest recall
prec, rec, thr = precision_recall_curve(y_treino, s_oof)
prec, rec = prec[:-1], rec[:-1]            # alinha com o vetor de limites / align with the thresholds vector
atende = prec >= PRECISAO_MINIMA

if atende.any():
    limite_final = thr[atende][np.argmax(rec[atende])]
    print(f"\nLimite escolhido (precisão >= {PRECISAO_MINIMA:.0%}, maior recall): {limite_final:.3f}")
else:
    # Plano B: se nenhum limite atinge a precisão mínima, usa o melhor F1
    # Plan B: if no threshold reaches the minimum precision, use the best F1
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    limite_final = thr[np.argmax(f1)]
    print(f"\nNenhum limite atinge {PRECISAO_MINIMA:.0%} de precisão. Usando o melhor F1: {limite_final:.3f}")

# **9. AVALIAÇÃO FINAL NO TESTE**

#### Agora sim uso o teste, uma única vez. Treino todos os modelos com o treino inteiro, comparo as curvas ROC e Precisão × Recall e aplico no melhor modelo o limite escolhido na validação cruzada.

In [ ]:
# Treina cada modelo com o treino inteiro e gera o score no teste
# Train every model on the full training set and score the test set
modelos_finais, scores_teste = {}, {}
for nome, modelo in modelos.items():
    m = clone(modelo)
    m.fit(X_treino, y_treino)
    modelos_finais[nome] = m
    scores_teste[nome] = m.predict_proba(X_teste)[:, 1]

# Tabela de métricas no teste (inclui o baseline de Bayes só com o valor)
# Test metrics table (includes the amount-only Bayes baseline)
resultados_teste = pd.DataFrame({
    nome: {"PR-AUC": average_precision_score(y_teste, s), "ROC-AUC": roc_auc_score(y_teste, s)}
    for nome, s in scores_teste.items()
}).T
resultados_teste.loc["Bayes (só valor)"] = [pr_valor, roc_valor]
resultados_teste = resultados_teste.round(4).sort_values("PR-AUC", ascending=False)
display(resultados_teste)
print("PR-AUC de um chute (priori):", round(y_teste.mean(), 4))

In [ ]:
# Curvas ROC e Precisão × Recall de todos os modelos, no teste
# ROC and Precision × Recall curves for all models, on the test set
roc_df, pr_df = [], []
for nome, s in scores_teste.items():
    fpr, tpr, _ = roc_curve(y_teste, s)
    p, r, _ = precision_recall_curve(y_teste, s)
    roc_df.append(pd.DataFrame({"FPR": fpr, "TPR": tpr, "Modelo": nome}))
    pr_df.append(pd.DataFrame({"Recall": r, "Precisão": p, "Modelo": nome}))
roc_df, pr_df = pd.concat(roc_df), pd.concat(pr_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=roc_df, x="FPR", y="TPR", hue="Modelo", estimator=None, sort=False, ax=ax1)
ax1.plot([0, 1], [0, 1], color="gray", linestyle="--")   # chute aleatório / random guess
ax1.set_xlabel("Taxa de falsos positivos (FPR)")
ax1.set_ylabel("Recall (TPR)")
ax1.set_title("Curva ROC")

sns.lineplot(data=pr_df, x="Recall", y="Precisão", hue="Modelo", estimator=None, sort=False, ax=ax2)
ax2.axhline(y_teste.mean(), color="gray", linestyle="--", label="Chute (priori)")
ax2.axhline(PRECISAO_MINIMA, color="black", linestyle=":", label="Precisão mínima")
ax2.set_title("Curva Precisão × Recall")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Aplica no teste o limite escolhido na validação cruzada
# Apply on the test set the threshold chosen via cross-validation
s_teste = scores_teste[melhor_modelo]
previsto = (s_teste >= limite_final).astype(int)
vn, fp, fn, vp = confusion_matrix(y_teste, previsto, labels=[0, 1]).ravel()

precisao_teste = vp / (vp + fp) if (vp + fp) else 0.0
recall_teste = vp / (vp + fn)
print(f"{melhor_modelo} | limite = {limite_final:.3f}")
print("  alertas dados:", vp + fp)
print("  alertas que eram fraude (vp):", vp)
print("  alertas falsos (fp):", fp)
print("  fraudes que passaram (fn):", fn)
print(f"  precisão: {precisao_teste:.2f} | recall: {recall_teste:.2f}")

# Matriz de confusão como heatmap
# Confusion matrix as a heatmap
matriz = pd.DataFrame([[vn, fp], [fn, vp]],
                      index=["Real: Normal", "Real: Fraude"],
                      columns=["Previsto: Normal", "Previsto: Fraude"])
plt.figure(figsize=(6, 4.5))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Matriz de confusão no teste ({melhor_modelo})")
plt.tight_layout()
plt.show()

# **10. VARIÁVEIS MAIS IMPORTANTES**

#### Ranking das 10 variáveis que mais contribuem para as decisões do melhor modelo de árvores (Random Forest ou CatBoost). A soma das importâncias de todas as variáveis dá 100%. Atenção: V1 a V28 vêm de uma transformação PCA, então sabemos *quais* pesam, mas não o que cada uma significa no mundo real.

In [ ]:
# Modelo de árvores a explicar: o melhor da CV se for de árvores; senão, o Random Forest
# Tree model to explain: the CV winner if it is tree-based; otherwise Random Forest
modelo_imp = melhor_modelo if melhor_modelo in ("Random Forest", "CatBoost") else "Random Forest"
m_imp = modelos_finais[modelo_imp]

importancia = (pd.Series(m_imp.feature_importances_, index=X_treino.columns)
               .sort_values(ascending=False).head(10).rename("importancia").reset_index()
               .rename(columns={"index": "variavel"}))

plt.figure(figsize=(8, 5))
sns.barplot(data=importancia, x="importancia", y="variavel", hue="variavel",
            palette="viridis", legend=False)
plt.title(f"Top 10 variáveis ({modelo_imp})")
plt.xlabel("Importância")
plt.ylabel("")
plt.tight_layout()
plt.show()

# **CONCLUSÃO**

#### O texto abaixo é gerado a partir dos resultados desta execução, então os números sempre batem com as tabelas acima.

In [ ]:
# Conclusão gerada automaticamente a partir dos resultados
# Conclusion generated automatically from the results
top4 = ", ".join(importancia["variavel"].head(4))
nb_pr = resultados_teste.loc["Naive Bayes", "PR-AUC"]
nb_roc = resultados_teste.loc["Naive Bayes", "ROC-AUC"]
pr_teste = resultados_teste.loc[melhor_modelo, "PR-AUC"]
pr_cv = resumo_cv.loc[melhor_modelo, "PR-AUC (média)"]
pr_cv_dp = resumo_cv.loc[melhor_modelo, "PR-AUC (desvio)"]

# Segundo colocado na CV: a diferença é maior que o ruído entre folds?
# Runner-up in CV: is the gap larger than the fold-to-fold noise?
ranking = resumo_cv["PR-AUC (média)"].sort_values(ascending=False)
segundo = ranking.index[1]
gap = ranking.iloc[0] - ranking.iloc[1]
ruido_top = max(resumo_cv.loc[melhor_modelo, "PR-AUC (desvio)"], resumo_cv.loc[segundo, "PR-AUC (desvio)"])
veredito = ("há vantagem acima do ruído entre folds" if gap > ruido_top
            else "a diferença está dentro do ruído entre folds, então na prática empatam")

print(f"""
1. Saber só o valor da transação quase não ajuda: PR-AUC de {pr_valor:.3f}
   contra {y_teste.mean():.3f} de um chute (o priori).
2. Usar todas as variáveis ajuda muito: o Naive Bayes chega a ROC-AUC {nb_roc:.2f},
   mas com PR-AUC de só {nb_pr:.2f}, ou seja, muitos alertas falsos.
3. Melhor modelo na validação cruzada: {melhor_modelo} (PR-AUC {pr_cv:.3f} ± {pr_cv_dp:.3f});
   no teste, PR-AUC {pr_teste:.3f}. Contra o {segundo}, {veredito}.
4. Com limite {limite_final:.3f} (escolhido sem olhar o teste, exigindo precisão >= {PRECISAO_MINIMA:.0%}),
   o modelo deu {vp + fp} alertas no teste: {vp} eram fraude e {fp} eram falsos.
   Precisão {precisao_teste:.2f} e recall {recall_teste:.2f} ({fn} fraudes passaram).
5. As variáveis que mais pesam ({modelo_imp}) são {top4}.
""")

#### **Próximos passos possíveis**
- Comparar McNemar (separado em fraudes e normais) e bootstrap pareado do PR-AUC entre os dois melhores modelos.
- Escolher o limite por custo financeiro, usando o `Amount` real de cada transação.
- Rodar o notebook com `REMOVER_DUPLICATAS = True` e comparar os resultados.
- Calibração das probabilidades, XGBoost/LightGBM e explicabilidade com SHAP.
- Split temporal (treinar no começo, testar no fim) e salvar o melhor modelo com `joblib`.